## MNIST — raqamlarni tanish (PyTorch)

Bu loyihada qo'lda yozilgan raqamlarni (0-9) tanib oladigan oddiy neural network quraman. Bu mening birinchi PyTorch loyiham, deep learning asoslarini (loss, optimizer, training loop) mustahkamlash uchun qildim.

Avval MNIST datasetini yuklab olaman — 28x28 piksel o'lchamdagi qo'lda yozilgan raqamlar rasmlari.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torchvision import datasets, transforms

train_data = datasets.MNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())
test_data = datasets.MNIST(root='./data', train=False, download=True, transform=transforms.ToTensor())

Datani kichik bo'laklarga (batch) bo'lib beradigan DataLoader yasayman — bu training paytida xotirani tejaydi va o'rganishni barqarorroq qiladi.

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

for images, labels in train_loader:
    print(images.shape)  # [64, 1, 28, 28]
    print(labels.shape)  # [64]
    break

Model — oddiy, bitta yashirin qatlamli fully-connected network. Rasm 28x28=784 pikseldan iborat, chiqishda esa 10 ta klass (0 dan 9 gacha raqamlar).

In [ ]:
model = nn.Sequential(
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
)

Training qismi. Birinchi urinishda `SGD` optimizer va 5 epoch bilan sinadim, natija yomon emas edi (test accuracy 91.48%), lekin keyin `Adam`ga o'tib, epoch sonini 10ga oshirganimda natija sezilarli yaxshilandi.

In [ ]:
loss = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 10
losses = []

for epoch in range(epochs):
    epoch_loss = 0
    for images, labels in train_loader:
        images = images.view(images.shape[0], -1)

        output = model(images)
        loss_value = loss(output, labels)
        epoch_loss += loss_value.item()

        optimizer.zero_grad()
        loss_value.backward()
        optimizer.step()
    losses.append(epoch_loss/len(train_loader))
    print(f"Epoch {epoch+1}/{epochs} Loss: {epoch_loss/len(train_loader)}")

plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training loss')
plt.show()

Training paytida loss pasayib borayotganini ko'rish yaxshi, lekin model haqiqatan qanchalik yaxshi ishlashini bilish uchun uni **ko'rmagan** data — test setda sinash kerak.

In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.view(images.shape[0], -1)

        output = model(images)
        predictions = torch.argmax(output, dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.shape[0]

accuracy = correct / total
print(f"Test accuracy: {accuracy * 100:.2f}%")

## Xulosa

| Sozlama | Test accuracy |
|---|---|
| SGD, 5 epoch | 91.48% |
| **Adam, 10 epoch** | **97.80%** |

Optimizer'ni SGD'dan Adam'ga almashtirish va epoch sonini oshirish natijani sezilarli yaxshiladi. Bu model juda sodda (bitta yashirin qatlam) bo'lishiga qaramay, 97.8% — MNIST uchun yomon natija emas. Keyingi qadam sifatida CNN (convolutional layer) qo'shib ko'rish mumkin, bu odatda 99%+ natija beradi.